# Silver 1: parse the PDFs and check each parse

Task `silver_parse` of the job `ingest_financebench`. Parses every **part** of a PDF (the PDFs cut into parts of a
few pages, see `split_pdfs`) that is still **missing**, with `ai_parse_document`, and checks each result. There is
no separate retry task: a part whose parse failed is simply still missing, so **the next run parses it again**.

| A part that is... | This run |
| --- | --- |
| Not parsed, never tried | Parses it |
| Failed before in a retryable way (rate limit, internal error), with attempts left | Parses it again |
| Failed before and cannot succeed (a failed conversion) | Leaves it alone. It stays in `silver_failures` |
| Failed `max_attempts` times | Leaves it alone. Raise `max_attempts` for one run to try again |
| Already in `silver_parsed` | Nothing. It is never parsed (or paid for) twice |

| Result of the parse | Goes to |
| --- | --- |
| Complete: it has elements and no page errors | `silver_parsed`, one row per part |
| Some pages failed (`Request rate limit exceeded`, `Internal error`) | `silver_log`, marked **retryable**. It is not stored: a part with missing pages would silently lose text. Only that part is parsed again, not the whole PDF |
| No elements: the part is over 500 pages (not possible after the split), or the conversion failed | `silver_log`, marked **not retryable** |

The checks are in [checks.py](./checks.py) (plain Python, with tests in `tests/`). The tables are defined in
[silver_tables.py](./silver_tables.py), and the parse-and-file function is in [parsing.py](./parsing.py).

**A part that fails is not tried again in the same run.** The list of missing parts is made once, at the start. A
part that fails goes to `silver_log` with the next attempt number, and the next run picks it up. A busy service is
not hit again straight away, and one run never loops.

**Load.** Parts are parsed `pdfs_per_batch` PDFs at a time (all the missing parts of those PDFs in one call), with
`batch_pause_seconds` between batches. Two batches in a row that store nothing stop the run: the service keeps
failing, run again later. A part that was never tried goes first, then the ones to retry, fewest attempts first.

**Why not a stream.** An earlier version read `bronze_pdf_parts` as a stream with a checkpoint. A stream sees each
row once, so a failed part could not come back and needed its own retry task. Now the tables say what is missing,
so a rerun, a retry and a recovery after a crash are the same code, and there is no checkpoint to keep.

**Challenge: `ai_parse_document` answers some pages with an error.** In the first full run 3,180 of 49,965 pages (6%),
in 55 PDFs, failed with `Request rate limit exceeded` or `Internal error`, because 10 PDFs were parsed at once.
**Solution:** split long PDFs into parts of 50 pages (a test PDF that lost 27 pages whole came back complete in
parts), check every parse, never store an incomplete one, parse two PDFs at a time, and parse the failures again
on the next run. A retry costs at most one part, which is `part_pages` pages.

In [ ]:
import sys

sys.path.insert(0, "../common")
import time

from names import Names
from parsing import parse_docs
from silver_tables import create_silver_tables

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("pdfs_per_batch", "2")  # PDFs parsed at once (all their missing parts). Small, so the parser's rate limit is not hit
dbutils.widgets.text("max_attempts", "3")  # a part is given up on after this many failed parses
dbutils.widgets.text("batch_pause_seconds", "5")  # pause between batches

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("bronze_schema"), dbutils.widgets.get("silver_schema"))
pdfs_per_batch = int(dbutils.widgets.get("pdfs_per_batch"))
max_attempts = int(dbutils.widgets.get("max_attempts"))
batch_pause_seconds = int(dbutils.widgets.get("batch_pause_seconds"))

create_silver_tables(spark, n)

In [ ]:
def missing_parts():
    """The parts to parse in this run, as [(path, [part_no, ...]), ...]: not in silver_parsed, and not given up on.

    A part is given up on when its last failure is not retryable, or it has failed max_attempts times."""
    rows = spark.sql(f"""
    SELECT t.path, t.part_no, coalesce(f.attempts, 0) AS attempts
    FROM {n.parts} t
    LEFT JOIN {n.parsed} p ON p.path = t.path AND p.part_no = t.part_no
    LEFT JOIN (
      SELECT path, part_no, max(attempt) AS attempts, max_by(retryable, attempt) AS retryable
      FROM {n.log} WHERE stage = 'parse' AND part_no IS NOT NULL GROUP BY path, part_no
    ) f ON f.path = t.path AND f.part_no = t.part_no
    WHERE p.path IS NULL AND (f.path IS NULL OR (f.retryable AND f.attempts < {max_attempts}))
    """).collect()
    by_pdf = {}
    for r in rows:
        by_pdf.setdefault(r.path, []).append((r.part_no, r.attempts))
    order = sorted(by_pdf, key=lambda path: (max(a for _, a in by_pdf[path]), path))  # never tried first
    return [(path, sorted(no for no, _ in by_pdf[path])) for path in order]

In [ ]:
todo = missing_parts()
print(f"{sum(len(nos) for _, nos in todo)} parts of {len(todo)} PDFs to parse.")

parsed, failed, nothing_in_a_row = 0, 0, 0
for i in range(0, len(todo), pdfs_per_batch):
    if nothing_in_a_row >= 2:  # the service keeps failing: stop, run again later
        print("Stopped: two batches in a row stored nothing, so the service keeps failing. Run again later.")
        break
    batch = todo[i:i + pdfs_per_batch]
    keys = spark.createDataFrame([(path, no) for path, nos in batch for no in nos], "path string, part_no int")
    good, bad = parse_docs(spark, n, spark.table(n.parts).join(keys, ["path", "part_no"]))
    parsed, failed = parsed + good, failed + bad
    nothing_in_a_row = nothing_in_a_row + 1 if good == 0 else 0
    print(f"PDFs {i + 1}-{i + len(batch)} of {len(todo)}: {good} parts parsed, {bad} logged as failed")
    time.sleep(batch_pause_seconds)
print(f"This run: {parsed} parts parsed, {failed} failed (the next run tries the retryable ones again).")

In [ ]:
print("Parts parsed (silver_parsed):", spark.table(n.parsed).count(), "of", spark.table(n.parts).count())
display(spark.table(n.failures).groupBy("retryable").count())